# 09. Gemini 기반 리뷰 감성분석

**목적**  
리뷰를 긍정·부정·애매로 분류하고 배송 관련 리뷰를 제외합니다.

**입력**  
`크롤링한 리뷰 데이터`

**출력**  
`data/processed의 Gemini 감성분석 결과`

> 기본값에서는 Gemini API를 호출하지 않고 저장된 전체 결과를 확인합니다.


In [ ]:
from pathlib import Path

# Jupyter와 Colab 모두 저장소 루트에서 실행합니다.
def find_project_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "data").is_dir() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError("저장소를 clone한 뒤 해당 폴더 안에서 실행하세요.")

PROJECT_ROOT = find_project_root()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [DATA_RAW_DIR, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 안전 기본값: 저장된 결과를 읽고 API를 호출하지 않습니다.
RUN_GEMINI_API = False
FULL_RUN = False
MAX_TEST_REVIEWS = 20
print("Gemini API 실행:", RUN_GEMINI_API)


## 감성분석 코드

API 분석 코드는 `RUN_GEMINI_API=True`로 명시한 경우에만 실행됩니다.

## 1. 라이브러리 설치 및 불러오기

이 부분에서는 필요한 파이썬 라이브러리들을 설치하고 불러옵니다. `google-genai`, `pandas`, `tqdm` 등을 설치하고 `os`, `getpass` 등의 모듈을 가져와 사용합니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    import os
    from getpass import getpass

    if RUN_GEMINI_API:
        api_key = os.getenv("GEMINI_API_KEY", "").strip()

        if not api_key:
            try:
                from google.colab import userdata
                api_key = (userdata.get("GEMINI_API_KEY") or "").strip()
            except Exception:
                api_key = ""

        if not api_key:
            api_key = getpass("Gemini API key: ").strip()

        if not api_key:
            raise ValueError("Gemini API 키가 필요합니다.")

        os.environ["GEMINI_API_KEY"] = api_key


## 2. Gemini API 키 설정

Gemini API를 사용하기 위해 API 키를 설정하는 부분입니다. `getpass()` 함수를 사용하여 사용자로부터 API 키를 안전하게 입력받아 환경 변수로 설정합니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    path1 = DATA_RAW_DIR / "올리브영_크림240개_피부타입_피부톤_리뷰.csv"


## 3. 데이터 불러오기

주어진 CSV 파일(`올리브영_크림240개_피부타입_피부톤_리뷰.csv`)을 `pandas` 라이브러리를 사용하여 데이터프레임으로 불러옵니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    import pandas as pd

    df = pd.read_csv(path1)


## 4. 데이터 초기 탐색

불러온 데이터의 기본적인 정보를 확인합니다. 전체 행 수, 컬럼 목록을 출력하고 데이터프레임의 첫 5개 행을 `display()` 함수로 보여줍니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    print("전체 행 수:", len(df))
    print("컬럼:", df.columns.tolist())

    display(df.head())


전체 행 수: 57903
컬럼: ['상품번호', '피부타입', '피부톤', '정렬', '평점', '리뷰내용']


,상품번호,피부타입,피부톤,정렬,평점,리뷰내용
0,A000000260257,지성,쿨톤,유용한순,5,너무 만족합니다 좋아요 굿!! 최고입니다 \r\n지성피부 잘 맞아요
1,A000000260257,지성,쿨톤,유용한순,5,자극없이순하고 데이케어용 보습으로 사용하기 좋아요. 화장 전에도 나름 괜찮은 것 같아요
2,A000000260257,지성,쿨톤,유용한순,5,🌱나 민감성 피부🌱\r\n\r\n피부 예민할 때 아무거나 못 바르잖아\r\n이건 그냥 편하게 손이 가는 크림이었음\r\n\r\n처음 바를 때 자극 느낌 거의 없고\r\n부드럽게 펴발려서 부담 없이 쓰기 좋았음\r\n\r\n막 엄청 꾸덕한 건 아닌데\r\n바르면 촉촉함은 바로 올라오는 편이고\r\n겉돌지 않고 자연스럽게 스며드는 느낌\r\n\r\n건조해서 당길 때 발라주면\r\n금방 편안해지는 느낌 들어서 괜찮았음\r\n\r\n유분감도 과하지 않아서\r\n지성인데도 크게 답답함 없었고\r\n번들거리기보단 깔끔하게 마무리되는 쪽\r\n\r\n와 이거 대박이다 이런 느낌보단\r\n피부 컨디션 안 좋을 때\r\n무난하게 계속 쓰게 되는 그런 크림임
3,A000000260257,지성,쿨톤,유용한순,5,"피부과에서 추천하는 크림만큼 피부 유수분을 잘맞춰주고\r\n기름짐이 없어요 지성 수부지분들은 정말 만족하실만한 크림\r\n유분이 안느껴지고 끈적거리지 않는 마무리감, 발림성이 너무 좋아요 단점이 없는 크림이라고 생각됩니다.\r\n저는 여름에는 막 써주고 가을 겨울에는 살짝 건조해 레이어드 몇번 해줘 발라줍니다.그럼 전혀 건조해 지지 않아요\r\n\r\n제형은 꾸덕하지 않고 수분감이 좀 있는 크림제형이라고 생각하시면 됩니다 흡수도 좋고 화장품 궁합도 잘 맞아 전혀 밀리지 않습니다 이번에 할인할때 많이 쟁이는중"
4,A000000260257,지성,쿨톤,유용한순,5,"예전부터 아토피문제로 피부과 가서 크림 처방 받던게 제로이드 였어요. 타 제품도 받아봤지만 저는 제로이드가 가장 발림성이 깔끔해서 좋더라구요. \r\n올영에 들어와서 완전 신세계인데, 수딩크림이어서 그런지\r\n발림성 굉장히 깔끔하고, 보습과 진정은 잘 돼요!\r\n아침 메이크업 전에도 발라보고 또 한달 후기 남길게요\r\n발림성도 찍어서 올려보아요 도움이 되셨다면 👍🏻"


## 5. 리뷰 내용 정제

`리뷰내용` 컬럼의 텍스트 데이터를 정제하는 과정입니다.

- `clean_review` 함수를 정의하여 줄바꿈, 탭, 연속된 공백 등을 제거하고 텍스트를 정리합니다.
- `isna().sum()`과 `.str.strip() == ""`를 통해 원본 데이터에 빈 리뷰나 결측치가 있었는지 확인합니다.
- 정제된 `리뷰내용`이 길이가 0인 행을 제거하고 인덱스를 재설정합니다.
- 최종 데이터 수와 중복을 제외한 고유 리뷰 수를 확인합니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    import re
    import numpy as np

    REVIEW_COLUMN = "리뷰내용"

    # 원래 데이터에 빈 리뷰나 결측치가 있었는지 먼저 확인해봅니다.
    original_empty_count = df[REVIEW_COLUMN].isna().sum() + (df[REVIEW_COLUMN].astype(str).str.strip() == "").sum()
    print(f"정제 전 빈 리뷰(또는 결측치) 개수: {original_empty_count}개\n")

    def clean_review(text):
        if pd.isna(text):
            return ""

        text = str(text)

        # 줄바꿈과 탭을 공백으로 변경
        text = re.sub(r"[\r\n\t]+", " ", text)

        # 연속 공백 정리
        text = re.sub(r"\s+", " ", text)

        return text.strip()


    df[REVIEW_COLUMN] = df[REVIEW_COLUMN].apply(clean_review)

    # 빈 리뷰 제거 (결과적으로 0개이므로 원본 데이터 수 유지)
    df = df[df[REVIEW_COLUMN].str.len() > 0].copy()
    df.reset_index(drop=True, inplace=True)

    print("빈 리뷰 제거 후 전체 데이터 수:", len(df))
    print("고유 리뷰 수 (중복 제외):", df[REVIEW_COLUMN].nunique())


정제 전 빈 리뷰(또는 결측치) 개수: 0개

빈 리뷰 제거 후 전체 데이터 수: 57903
고유 리뷰 수 (중복 제외): 50444


## 6. Gemini 감성 분석 시스템 프롬프트 정의

Gemini 모델이 리뷰를 감성 분석할 때 따를 상세한 지침(판단 기준, 중요 규칙, 예시)을 `SYSTEM_PROMPT_3` 변수에 문자열로 정의합니다. 모델은 이 프롬프트를 기반으로 '긍정', '부정', '애매' 중 하나를 출력하도록 지시받습니다.

In [ ]:
if RUN_GEMINI_API:
    pass


In [ ]:
if RUN_GEMINI_API:
    SYSTEM_PROMPT_3 = """
    당신은 화장품 리뷰 감성 분석 전문가입니다.

    사용자가 작성한 리뷰를 읽고 제품에 대한 전반적인 만족도를 판단하세요.

    출력은 반드시 아래 셋 중 하나만 출력하세요.

    긍정
    부정
    애매

    설명, 이유, 추가 문장은 절대 출력하지 마세요.

    ========================
    판단 기준
    ========================

    [긍정]

    다음 중 하나라도 해당하면 긍정입니다.

    - 제품에 만족한다.
    - 사용할 만하다.
    - 괜찮다.
    - 나쁘지 않다.
    - 무난하다.
    - 재구매 의사가 있다.
    - 추천한다.
    - 특정 피부 타입에 잘 맞는다.
    - 특정 계절이나 상황에서 사용하기 좋다고 평가한다.
    - 일부 단점이 있지만 전체적으로 만족한다.

    [부정]

    다음 중 하나라도 해당하면 부정입니다.

    - 제품에 실망했다.
    - 효과를 거의 느끼지 못했다.
    - 별로다.
    - 만족하지 않는다.
    - 재구매 의사가 없다.
    - 추천하지 않는다.
    - 단점 때문에 전반적으로 만족하지 않는다.

    [애매]

    아래 경우에만 애매를 선택하세요.

    1. 아직 사용하지 않았다.
    2. 효과를 아직 모르겠다.
    3. 제품 평가 없이 배송, 가격, 포장만 언급했다.
    4. 제품 만족도를 판단할 정보가 없다.

    ========================
    중요 규칙
    ========================

    1.

    피부 고민(건조함, 유분, 민감성, 트러블 등)을 설명하는 것은 부정이 아닙니다.

    2.

    특정 피부 타입이나 계절에 적합하다고 평가하면 긍정입니다.

    예)
    - 여름에 사용하기 좋다.
    - 화장 전에 사용하기 좋다.
    - 지성 피부에 잘 맞는다.
    - 데일리로 쓰기 좋다.

    → 긍정

    3.

    장점과 단점이 함께 있으면

    최종 만족도를 기준으로 판단하세요.

    예)

    "보습은 조금 아쉽지만 전체적으로 만족한다."

    → 긍정

    "발림은 좋지만 재구매는 안 할 것 같다."

    → 부정

    4.

    "나쁘지 않다"

    "괜찮다"

    "무난하다"

    "사용할 만하다"

    → 긍정입니다.

    5.

    애매는 가장 마지막 선택입니다.

    긍정 또는 부정으로 판단할 수 있다면 애매를 선택하지 마세요.

    ========================
    예시
    ========================

    리뷰:
    촉촉하고 흡수도 빨라서 만족합니다. 재구매할 예정입니다.

    결과:
    긍정


    리뷰:
    여름에 가볍게 사용하기 좋고 데일리로 쓰기 괜찮습니다.

    결과:
    긍정


    리뷰:
    발림성은 좋지만 보습은 조금 부족합니다. 그래도 무난하게 사용할 만합니다.

    결과:
    긍정


    리뷰:
    효과를 잘 모르겠고 재구매는 안 할 것 같습니다.

    결과:
    부정


    리뷰:
    기대보다 별로였고 트러블도 생겨서 다시는 안 살 것 같습니다.

    결과:
    부정


    리뷰:
    아직 사용 전이라 효과는 모르겠습니다.

    결과:
    애매


    리뷰:
    배송이 빨랐습니다.

    결과:
    애매
    """


## 7. Gemini 클라이언트 초기화 및 모델 테스트

- `google.genai` 라이브러리를 사용하여 Gemini API 클라이언트를 초기화합니다.
- `SentimentItem`과 `SentimentBatch` Pydantic 모델을 정의하여 Gemini API 응답의 구조를 명확히 합니다.
- 여러 Gemini 모델 버전(`gemini-2.5-flash-lite`, `gemini-2.0-flash-lite`, `gemini-3.1-flash-lite`, `gemini-flash-lite-latest`)에 대해 간단한 테스트 요청을 보내 사용 가능한 모델을 확인합니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    from google import genai
    from google.genai import types
    from pydantic import BaseModel
    from typing import Literal, List
    import os

    client = genai.Client(
        api_key=os.environ["GEMINI_API_KEY"]
    )


    class SentimentItem(BaseModel):
        id: int
        sentiment: Literal["긍정", "부정", "애매"]


    class SentimentBatch(BaseModel):
        results: List[SentimentItem]


    MODEL_NAME = "gemini-3.1-flash-lite"

    print("Gemini API 클라이언트가 성공적으로 초기화되었습니다.")


Gemini API 클라이언트가 성공적으로 초기화되었습니다.


## 8. `analyze_review_batch` 함수 정의

이 함수는 주어진 리뷰 데이터 배치에 대해 Gemini API를 호출하여 감성 분석을 수행합니다.

- 입력 데이터의 필수 컬럼(`review_id`, `리뷰내용`)을 확인하고 `review_id`의 중복 여부를 검사합니다.
- 각 리뷰를 JSON 형식으로 변환하여 Gemini 모델에 전달할 프롬프트를 생성합니다.
- `client.models.generate_content`를 호출하여 Gemini API와 통신하며, `system_instruction`에 `SYSTEM_PROMPT_3`을, `response_mime_type`에 `application/json`을, `response_schema`에 `SentimentBatch`를 사용하여 구조화된 JSON 응답을 받도록 설정합니다.
- API 호출 실패 시 `max_retries`만큼 재시도 로직을 포함하여 안정성을 높였습니다.
- 응답으로 받은 감성 분석 결과(`긍정`, `부정`, `애매`)를 검증하고 `review_id`를 키로 하는 딕셔너리 형태로 반환합니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    import json
    import time
    import random
    from collections import Counter

    VALID_LABELS = {"긍정", "부정", "애매"}


In [ ]:
if RUN_GEMINI_API:
    def analyze_review_batch(
        batch_df,
        max_retries=5
    ):
        """
        Gemini를 사용해 리뷰 한 배치를 감성 분석합니다.

        필수 컬럼:
        - review_id
        - 리뷰내용

        반환:
        {
            review_id: "긍정" | "부정" | "애매",
            ...
        }
        """

        if batch_df.empty:
            return {}

        required_columns = {"review_id", "리뷰내용"}
        missing_columns = required_columns - set(batch_df.columns)

        if missing_columns:
            raise ValueError(
                f"필수 컬럼이 없습니다: {sorted(missing_columns)}"
            )

        batch_df = batch_df.copy()
        batch_df["review_id"] = batch_df["review_id"].astype(int)

        duplicated_input_ids = batch_df.loc[
            batch_df["review_id"].duplicated(),
            "review_id"
        ].tolist()

        if duplicated_input_ids:
            raise ValueError(
                f"입력 데이터에 중복 review_id가 있습니다: "
                f"{duplicated_input_ids[:20]}"
            )

        batch_df["리뷰내용"] = (
            batch_df["리뷰내용"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        review_data = [
            {
                "id": int(row.review_id),
                "review": row.리뷰내용
            }
            for row in batch_df.itertuples(index=False)
        ]

        expected_ids = set(batch_df["review_id"].tolist())

        last_error = None
        retry_instruction = ""

        for attempt in range(1, max_retries + 1):

            try:
                current_prompt = f"""
    아래 화장품 리뷰를 각각 독립적으로 분석하세요.

    각 리뷰에 대해 반드시 다음 세 라벨 중 하나만 선택하세요.

    - 긍정
    - 부정
    - 애매

    반드시 지켜야 할 출력 규칙:

    1. 입력된 모든 id를 정확히 한 번씩 반환하세요.
    2. 총 {len(review_data)}개의 결과를 반환하세요.
    3. id 값을 변경하지 마세요.
    4. 입력에 없는 id를 추가하지 마세요.
    5. 리뷰 순서와 관계없이 각 리뷰를 독립적으로 판단하세요.
    6. 설명이나 판단 근거는 출력하지 마세요.
    7. system instruction의 감성 판단 기준을 따르세요.
    8. 응답 생성 전에 모든 입력 id가 결과에 포함됐는지 확인하세요.

    {retry_instruction}

    리뷰 목록:

    {json.dumps(review_data, ensure_ascii=False)}
    """.strip()

                response = client.models.generate_content(
                    model=MODEL_NAME,
                    contents=current_prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT_3 ,
                        temperature=0,
                        max_output_tokens=max(
                            2048,
                            len(batch_df) * 30
                        ),
                        thinking_config=types.ThinkingConfig(
                            thinking_budget=0
                        ),
                        response_mime_type="application/json",
                        response_schema=SentimentBatch,
                    )
                )

                if response.parsed is not None:

                    if isinstance(response.parsed, SentimentBatch):
                        parsed_result = response.parsed
                    else:
                        parsed_result = SentimentBatch.model_validate(
                            response.parsed
                        )

                else:
                    if not response.text:
                        raise ValueError(
                            "Gemini 응답 내용이 비어 있습니다."
                        )

                    parsed_json = json.loads(response.text)

                    parsed_result = SentimentBatch.model_validate(
                        parsed_json
                    )

                result_items = parsed_result.results

                returned_ids = [
                    int(item.id)
                    for item in result_items
                ]

                id_counts = Counter(returned_ids)

                duplicate_result_ids = sorted(
                    review_id
                    for review_id, count in id_counts.items()
                    if count > 1
                )

                if duplicate_result_ids:
                    raise ValueError(
                        f"응답에 중복된 ID가 있습니다: "
                        f"{duplicate_result_ids}"
                    )

                received_ids = set(returned_ids)

                missing_ids = expected_ids - received_ids
                unexpected_ids = received_ids - expected_ids

                if missing_ids:
                    raise ValueError(
                        f"결과 누락 ID: {sorted(missing_ids)}"
                    )

                if unexpected_ids:
                    raise ValueError(
                        f"입력에 없는 ID가 반환됨: "
                        f"{sorted(unexpected_ids)}"
                    )

                if len(result_items) != len(batch_df):
                    raise ValueError(
                        f"결과 개수 불일치: "
                        f"입력 {len(batch_df)}개 / "
                        f"출력 {len(result_items)}개"
                    )

                result_dict = {}

                for item in result_items:

                    review_id = int(item.id)
                    sentiment = str(item.sentiment).strip()

                    if sentiment not in VALID_LABELS:
                        raise ValueError(
                            f"허용되지 않은 라벨: "
                            f"id={review_id}, label={sentiment}"
                        )

                    result_dict[review_id] = sentiment

                return result_dict

            except Exception as e:

                last_error = e

                retry_instruction = (
                    f"이전 응답은 검증에 실패했습니다.\n"
                    f"오류: {type(e).__name__}: {e}\n"
                    f"이번 응답에서는 반드시 전체 "
                    f"{len(review_data)}개 id를 누락 없이 반환하세요."
                )

                print(
                    f"재시도 {attempt}/{max_retries} | "
                    f"{type(e).__name__}: {e}"
                )

                if attempt < max_retries:

                    wait_time = min(
                        60,
                        (2 ** attempt) + random.uniform(0, 1)
                    )

                    time.sleep(wait_time)

        raise RuntimeError(
            f"배치 분석 최종 실패: {last_error}"
        )


## 9. 대규모 감성 분석 실행 및 저장

`analyze_review_batch` 함수를 사용하여 전체 고유 리뷰 데이터에 대해 감성 분석을 수행하고 중간 결과를 저장하는 과정입니다.

- `unique_reviews` 데이터프레임을 생성하고 `review_id`와 `리뷰분류` 컬럼을 추가합니다.
- `CHECKPOINT_PATH`에 저장된 기존 체크포인트 파일이 있으면 불러와서 이어서 작업을 진행할 수 있도록 합니다.
- `TEST_MODE` 설정에 따라 일부 데이터(100개) 또는 전체 데이터에 대해 분석을 진행합니다.
- `tqdm`을 사용하여 진행률을 시각적으로 표시하며, `BATCH_SIZE`만큼 데이터를 나눠 배치 처리합니다.
- 각 배치 처리 후 `REQUEST_DELAY`만큼 지연 시간을 두어 API 요청 과부하를 방지합니다.
- 매 배치마다 `unique_reviews`를 CSV 파일(`gemini_리뷰감성분석_checkpoint.csv`)로 저장하여 중간 결과를 보존합니다.
- 오류 발생 시 `failed_batches` 리스트에 기록하고 재시도합니다.
- 모든 분석이 완료되면 최종 감성 분류 결과의 개수를 출력합니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    from tqdm.auto import tqdm
    from pathlib import Path
    import pandas as pd
    import time
    import os


    # =========================
    # 실행 설정
    # =========================

    BATCH_SIZE = 50

    # API 요청 사이 짧은 간격
    REQUEST_DELAY = 0.3

    # 중간 결과 저장 위치
    CHECKPOINT_PATH = DATA_INTERIM_DIR / "gemini_리뷰감성분석_checkpoint.csv"

    # 최종 파일 저장 위치
    FINAL_PATH = DATA_INTERIM_DIR / "올리브영_크림240개_리뷰감성분석_중간결과.csv"

    # True이면 100개만 실행
    # 전체 실행할 때 False로 변경
    TEST_MODE = False

    TEST_COUNT = 100


    # =========================
    # 고유 리뷰 데이터 생성
    # =========================

    unique_reviews = (
        df[[REVIEW_COLUMN]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    unique_reviews["review_id"] = unique_reviews.index
    unique_reviews["리뷰분류"] = pd.NA


    # =========================
    # 기존 결과 불러오기
    # =========================

    checkpoint_file = Path(CHECKPOINT_PATH)


In [ ]:
if RUN_GEMINI_API:
    if checkpoint_file.exists():
        previous = pd.read_csv(
            CHECKPOINT_PATH,
            encoding="utf-8-sig"
        )

        previous = previous[
            ["review_id", REVIEW_COLUMN, "리뷰분류"]
        ].copy()

        previous["review_id"] = previous["review_id"].astype(int)

        saved_map = dict(
            zip(
                previous["review_id"],
                previous["리뷰분류"]
            )
        )

        unique_reviews["리뷰분류"] = (
            unique_reviews["review_id"]
            .map(saved_map)
        )

        completed_count = (
            unique_reviews["리뷰분류"]
            .isin(VALID_LABELS)
            .sum()
        )

        print(f"기존 완료 결과: {completed_count:,}개")

    else:
        print("기존 체크포인트 없음: 처음부터 시작")


    # =========================
    # 테스트 또는 전체 범위 설정
    # =========================

    if TEST_MODE:
        target_ids = set(
            unique_reviews.head(TEST_COUNT)["review_id"]
        )
    else:
        target_ids = set(unique_reviews["review_id"])


    pending_mask = (
        unique_reviews["review_id"].isin(target_ids)
        & ~unique_reviews["리뷰분류"].isin(VALID_LABELS)
    )

    pending_df = unique_reviews.loc[
        pending_mask,
        ["review_id", REVIEW_COLUMN]
    ].copy()

    print(f"전체 고유 리뷰: {len(unique_reviews):,}개")
    print(f"이번 실행 대상: {len(target_ids):,}개")


In [ ]:
if RUN_GEMINI_API:
    print(f"이미 완료: {len(target_ids) - len(pending_df):,}개")
    print(f"남은 리뷰: {len(pending_df):,}개")


    # =========================
    # 감성 분석 실행
    # =========================

    failed_batches = []

    batch_starts = range(
        0,
        len(pending_df),
        BATCH_SIZE
    )


In [ ]:
if RUN_GEMINI_API:
    for start in tqdm(
        batch_starts,
        total=(len(pending_df) + BATCH_SIZE - 1) // BATCH_SIZE
    ):
        batch = pending_df.iloc[
            start:start + BATCH_SIZE
        ].copy()

        try:
            result_dict = analyze_review_batch(batch)

            batch_ids = batch["review_id"].astype(int)

            update_mask = unique_reviews[
                "review_id"
            ].isin(batch_ids)

            unique_reviews.loc[
                update_mask,
                "리뷰분류"
            ] = (
                unique_reviews.loc[
                    update_mask,
                    "review_id"
                ].map(result_dict)
            )

        except Exception as e:
            print(
                f"\n배치 실패 | "
                f"{start}~{start + len(batch) - 1} | {e}"
            )

            failed_batches.append(
                {
                    "start": start,
                    "review_ids": batch[
                        "review_id"
                    ].tolist(),
                    "error": str(e)
                }
            )

        # 매 배치마다 저장
        unique_reviews.to_csv(
            CHECKPOINT_PATH,
            index=False,
            encoding="utf-8-sig"
        )

        time.sleep(REQUEST_DELAY)


    print("\n감성 분석 실행 완료")
    print(
        unique_reviews["리뷰분류"]
        .value_counts(dropna=False)
    )


In [ ]:
if RUN_GEMINI_API:
    if failed_batches:
        print(f"실패 배치 수: {len(failed_batches)}")
    else:
        print("실패 배치 없음")


## 10. 분류 결과 확인

- `pd.set_option('display.max_colwidth', None)`를 설정하여 긴 리뷰 내용이 잘리지 않고 모두 표시되도록 합니다.
- '부정'으로 분류된 리뷰 내용을 확인합니다. 이를 통해 모델이 '부정'으로 판단한 리뷰들이 어떤 특징을 가지는지 볼 수 있습니다.
- '애매'로 분류된 리뷰 내용을 확인합니다. 이를 통해 모델이 '애매'로 판단한 리뷰들이 어떤 특징을 가지는지 볼 수 있습니다.
- `review_id` 컬럼은 불필요하므로 `drop(columns='review_id', inplace=True)`를 사용하여 제거합니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    import pandas as pd
    pd.set_option('display.max_colwidth', None)
    unique_reviews.loc[unique_reviews['리뷰분류'] == '부정', ['리뷰내용']]


,리뷰내용
8,정말 자극없이 순하긴 한데 마무리감이 약간 찝찝 거린다고 해야하나 그래서 ㅠ…번들거리지 않아서 좋지만 화장이랑은 잘 맞지 않을거슈같아요 대신 순힌거로는 정말 좋아요
20,평소에 화장품 아무거나 써도 뭐 안나는 강철 피부인데 이거만 바르면 얼굴 엄청 간지럽고 울긋불긋 트러블 바로 올라와요;;; 다른사람은 괜찮길래 새거 그냥 줬어요.. 제가 안맞는거겠죠 뭐.. 처음쓰실때 조심하세요
21,순한 크림이고 수부지 추천이어서 제로이드 수딩 크림을 사보았는데 환절기에 수부지가 바르기에는 다소 건조한 느낌이었습니다. 크림을 바르고 나면 아예 유분감이 느껴지지 않는 맘에 들었는데 건조한 얼굴에 수분을 채워주는 느낌은 없었습니다. 그리고 세콜지와 같이 피부 보습 장벽을 해주는 성분도 들어있지 않아서 여름쯤에는 가볍게 잘 사용할 것 같습니다!
22,자극은 없는 것 같은데 이거 발라도 조금 건조한 느낌? 살짝 아쉬워요
23,하 뭔가.. 이거 쓰고 피부 장벽 개선됐다! 이건 아니에여ㅠ 그냥 더 뭔가가 올라오지 않는다 정도?
...,...
50402,사기는 했는데 실제로 써보니 만족감 떨어져서 바로 처분했어요....
50403,생각보다 써보니 좋지는 않아서 괜히 샀네요~ 그래도 브랜드는 좋으니 콜라겐 팩이 좋아요~~ 그거는 추천이요
50426,성분이 자극적인건 전혀없는데 여름이라 습하기도하고 좀 찐득한 제형이라서 오히려 저는 티존만 지성인 복합성인데 이마에 트러블이 3년만에...낫어요ㅠ(좁쌀처럼작은염증성붉은트러블) 수딩크림말고 그 유명한 그냥 아토베리어크림이 훨 나아요 걍 기초바꿔보고싶어서 바꿨는데 피부복구시키는데 거의 10일 걸렸어요ㅜ...그래서 다시 흰색 근본템 쓰고있어용 그건 짱 좋아요 👍
50433,에스트라 제품 ... 몇년만에 궁금해서 다시 사봤는데 역시나 그냥 그렇다 ㅠ 재구매 의사 없음


In [ ]:
if RUN_GEMINI_API:
    pass
    import pandas as pd
    pd.set_option('display.max_colwidth', None)
    unique_reviews.loc[unique_reviews['리뷰분류'] == '애매', ['리뷰내용']]


,리뷰내용
148,저는 개인적으로 이거 피지 빨리 찬다고 생각하는데 ( 화이트헤드가 빨리 생긴다고요) 남동생은 똑같이 성인 여드름 피부인데 이거 아니면 안 된다고 하네요. 뭔가 여드름에도 종류가 다른가 봐요.
181,"최저가 찾아보고 구매하는 편인데, 올영이 제일 좋은 구성에 좋은 가격이었어요. 피부과에서도 추천할 정도로 믿고 쓰는 제로이드라, 인텐시브 써보고 살짝 무거운가 싶어서 수딩으로 구매해봤어요. 사용해보고 한달후기 남길게요~~~!"
196,🌸독자 개발 피부장벽 기술 MLE®️로 피부장벽을 튼튼하게! 🌸제로이드만의 독자 성분 디펜사마이드™️로 피부 방어력 UP 🌸피부 자극도 0.00으로 케어하는 민감피부 맞춤 피부 과학 솔루션 🌸민감 피부도 자극 걱정 ZERO 저자극 안심 포뮬라
242,"생각했던 수딩크림의 제형이 아니긴 한데, 수분기만 가득한 것보단 유분감이 살짝 있는 게 나은 것 같기도 하네요..! 조금 더 써봐야 알겠지만, 가격이 다소 비싼 편이라 다른 제품에 비해 확실히 나은 점이 있어야 할 것 같아요. 한 달 사용해보고 더 사용할지 말지 판단해보렵니다..!"
246,수분크림 이것저것 써보고 잇는데. 리뷰 조킬래 매장에서 구경하다 사밧어요
...,...
50266,비타 앰플 몇 통 째 잘 쓰고 있는 사람이라 크림도 사봤어요! 같이 쓰면 효과 좋다고 하니 꾸준히 써보고 후기 남기겠습미당
50272,세일기간에 구매해서 평소보다 저렴하게 구매한 이니스프리 잡티케어.
50289,앰플 사용 중인데 크림도 궁금해서 같이 구매했어용 기대되네용
50291,엄마가 수분크림이 필요하다고 해서 같이 가서 골라봤는데 아직까진 잘 모르겠다고 하시네용 좀 더 써보고 리뷰 쓰러 올게용


In [ ]:
if RUN_GEMINI_API:
    pass
    unique_reviews[unique_reviews['리뷰분류'] == '애매'].index.tolist()


[148,
 181,
 196,
 242,
 246,
 396,
 444,
 591,
 632,
 668,
 672,
 675,
 741,
 889,
 908,
 950,
 1048,
 1089,
 1531,
 1532,
 1533,
 1538,
 1539,
 1543,
 1582,
 1588,
 1589,
 1621,
 1630,
 1673,
 1675,
 1694,
 1715,
 1717,
 1736,
 1742,
 1744,
 1823,
 1840,
 1860,
 1894,
 1900,
 1901,
 2223,
 2268,
 2280,
 2283,
 2287,
 2310,
 2313,
 2378,
 2443,
 2469,
 2476,
 2498,
 2547,
 2549,
 2585,
 2689,
 3047,
 3268,
 3303,
 3364,
 3410,
 3416,
 3424,
 3426,
 3639,
 3676,
 3695,
 3759,
 3812,
 3818,
 3847,
 3879,
 3897,
 3912,
 3998,
 4051,
 4178,
 4182,
 4254,
 4298,
 4327,
 4356,
 4543,
 4544,
 4561,
 4584,
 4593,
 4610,
 4616,
 4677,
 4737,
 4760,
 4806,
 4809,
 4844,
 4947,
 4952,
 4971,
 4992,
 5004,
 5020,
 5026,
 5030,
 5038,
 5073,
 5087,
 5170,
 5171,
 5172,
 5199,
 5204,
 5208,
 5217,
 5218,
 5338,
 5342,
 5343,
 5396,
 5441,
 5491,
 5494,
 5496,
 5505,
 5556,
 5567,
 5625,
 5671,
 5678,
 5683,
 5723,
 5735,
 5827,
 5828,
 5833,
 5874,
 5883,
 5904,
 5905,
 5978,
 5980,
 5981,
 6221,
 

In [ ]:
if RUN_GEMINI_API:
    pass
    unique_reviews[unique_reviews['리뷰분류'] == '긍정'].index.tolist()


[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 24,
 25,
 26,
 27,
 28,
 29,
 31,
 32,
 33,
 34,
 35,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 87,
 88,
 89,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 103,
 104,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 136,
 137,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 162,
 163,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 175,
 177,
 178,
 179,
 180,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 207,
 208,
 

## 11. 원본 데이터에 감성 분석 결과 병합

원본 `df` 데이터에서 중복된 리뷰를 제거한 뒤, 감성 분석을 통해 얻은 `리뷰분류` 결과를 병합하는 과정입니다.

- `df.drop_duplicates(subset=['리뷰내용'])`을 사용하여 중복 리뷰를 제거합니다.
- `merge()` 함수를 사용하여 `unique_reviews`의 `리뷰내용`과 `리뷰분류` 컬럼을 `how='left'` 방식으로 병합합니다.
- 최종 데이터의 행 수와 `리뷰분류`가 null인 데이터의 개수를 확인하고, 고유한 상품 번호의 개수를 출력합니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    unique_reviews[unique_reviews['리뷰분류'] == '부정'].index.tolist()


[8,
 20,
 21,
 22,
 23,
 30,
 36,
 57,
 86,
 90,
 102,
 105,
 106,
 115,
 135,
 138,
 139,
 161,
 164,
 174,
 176,
 182,
 195,
 230,
 231,
 232,
 248,
 258,
 260,
 283,
 285,
 296,
 390,
 391,
 392,
 433,
 454,
 462,
 463,
 499,
 552,
 593,
 606,
 626,
 658,
 680,
 686,
 692,
 697,
 733,
 734,
 738,
 757,
 762,
 798,
 818,
 828,
 832,
 871,
 916,
 919,
 927,
 931,
 942,
 943,
 956,
 963,
 994,
 995,
 1017,
 1040,
 1049,
 1050,
 1079,
 1088,
 1091,
 1095,
 1102,
 1116,
 1117,
 1147,
 1158,
 1160,
 1166,
 1600,
 1696,
 1739,
 1772,
 1801,
 1802,
 1825,
 1826,
 1835,
 1865,
 1870,
 1871,
 1891,
 1924,
 1934,
 1940,
 1951,
 1964,
 1965,
 1967,
 1970,
 1994,
 2042,
 2045,
 2088,
 2127,
 2128,
 2163,
 2166,
 2179,
 2212,
 2231,
 2237,
 2238,
 2242,
 2259,
 2277,
 2297,
 2302,
 2305,
 2307,
 2324,
 2341,
 2351,
 2352,
 2363,
 2367,
 2411,
 2417,
 2419,
 2440,
 2441,
 2442,
 2451,
 2465,
 2473,
 2499,
 2502,
 2503,
 2525,
 2529,
 2566,
 2567,
 2568,
 2613,
 2679,
 2686,
 2937,
 3025,
 3050,
 3

In [ ]:
if RUN_GEMINI_API:
    pass
    unique_reviews.drop(columns='review_id',inplace=True)


In [ ]:
if RUN_GEMINI_API:
    pass
    unique_reviews.iloc[393,1]


'긍정'

In [ ]:
if RUN_GEMINI_API:
    pass
    unique_reviews.iloc[401,0]


'ㅇㄴ이거속건조왜케잘잡아줌?평소에크림써도건조햇는데이거쓰니까건조한거1도없이트로블이진정됏어요 ㄹㅇ미쳣다'

In [ ]:
if RUN_GEMINI_API:
    pass
    # 원본 데이터(df)에서 중복된 리뷰를 제거한 뒤, 분석 결과(리뷰분류)를 합쳐줍니다.
    # 이렇게 하면 unique_reviews와 동일한 행 수를 유지하면서 상품번호 등의 컬럼도 가져올 수 있습니다.

    final_df = df.drop_duplicates(subset=['리뷰내용']).merge(unique_reviews[['리뷰내용', '리뷰분류']], on='리뷰내용', how='left')

    print("최종 행 수:", len(final_df))
    display(final_df)


최종 행 수: 50444


,상품번호,피부타입,피부톤,정렬,평점,리뷰내용,리뷰분류
0,A000000260257,지성,쿨톤,유용한순,5,너무 만족합니다 좋아요 굿!! 최고입니다 지성피부 잘 맞아요,긍정
1,A000000260257,지성,쿨톤,유용한순,5,자극없이순하고 데이케어용 보습으로 사용하기 좋아요. 화장 전에도 나름 괜찮은 것 같아요,긍정
2,A000000260257,지성,쿨톤,유용한순,5,🌱나 민감성 피부🌱 피부 예민할 때 아무거나 못 바르잖아 이건 그냥 편하게 손이 가는 크림이었음 처음 바를 때 자극 느낌 거의 없고 부드럽게 펴발려서 부담 없이 쓰기 좋았음 막 엄청 꾸덕한 건 아닌데 바르면 촉촉함은 바로 올라오는 편이고 겉돌지 않고 자연스럽게 스며드는 느낌 건조해서 당길 때 발라주면 금방 편안해지는 느낌 들어서 괜찮았음 유분감도 과하지 않아서 지성인데도 크게 답답함 없었고 번들거리기보단 깔끔하게 마무리되는 쪽 와 이거 대박이다 이런 느낌보단 피부 컨디션 안 좋을 때 무난하게 계속 쓰게 되는 그런 크림임,긍정
3,A000000260257,지성,쿨톤,유용한순,5,"피부과에서 추천하는 크림만큼 피부 유수분을 잘맞춰주고 기름짐이 없어요 지성 수부지분들은 정말 만족하실만한 크림 유분이 안느껴지고 끈적거리지 않는 마무리감, 발림성이 너무 좋아요 단점이 없는 크림이라고 생각됩니다. 저는 여름에는 막 써주고 가을 겨울에는 살짝 건조해 레이어드 몇번 해줘 발라줍니다.그럼 전혀 건조해 지지 않아요 제형은 꾸덕하지 않고 수분감이 좀 있는 크림제형이라고 생각하시면 됩니다 흡수도 좋고 화장품 궁합도 잘 맞아 전혀 밀리지 않습니다 이번에 할인할때 많이 쟁이는중",긍정
4,A000000260257,지성,쿨톤,유용한순,5,"예전부터 아토피문제로 피부과 가서 크림 처방 받던게 제로이드 였어요. 타 제품도 받아봤지만 저는 제로이드가 가장 발림성이 깔끔해서 좋더라구요. 올영에 들어와서 완전 신세계인데, 수딩크림이어서 그런지 발림성 굉장히 깔끔하고, 보습과 진정은 잘 돼요! 아침 메이크업 전에도 발라보고 또 한달 후기 남길게요 발림성도 찍어서 올려보아요 도움이 되셨다면 👍🏻",긍정
...,...,...,...,...,...,...,...
50439,A000000260462,트러블성,쿨톤,유용한순,5,수부지라 개기름 올라오는 곳엔 잔뜩 올라오고 건조한 곳은 ㄹㅇ 갈라지고 이래서 고민했는데 매장 직원붐이 추천해주셔서 테스트 하고 바로 구매했어오 .. 기존 아토베리어는 여름에 쓰기엔 너무 부담이엏는데 이건 좋은듯,긍정
50440,A000000260462,트러블성,쿨톤,유용한순,5,에스트라 수딩크림이 안보이더니 리뉴얼되려고 안보였구나~ 똑 떨어져서 사려고 했는데 딱 잘 샀네요. 수딩크림 가벼움 +아토베리어 크림 보습감 섞인것같네요.이제 막 쓰기시작했으니 믿고씁니딘,긍정
50441,A000000260462,트러블성,봄웜톤,유용한순,5,다른 수분크림보다 자극적인거 같아요 비싼데 잘 모르겠어요ㅜ,부정
50442,A000000260462,트러블성,봄웜톤,유용한순,5,순하고 촉촉해서 좋아요!! 유분기 많이 없고 좋음 순해서 추천해요,긍정


## 12. '애매' 리뷰 및 배송 관련 리뷰 필터링

- `final_df`에서 '애매'로 분류된 리뷰를 제외하고 '긍정'과 '부정' 리뷰만 남겨 `final_df_pf`를 생성합니다.
- `reset_index(inplace=True)`를 사용하여 인덱스를 재설정합니다.
- `shipping_keywords` 리스트를 정의하여 배송과 관련된 키워드를 포함하는 리뷰를 식별합니다.
- `리뷰내용`에 `shipping_keywords` 중 하나라도 포함된 리뷰를 필터링하여 `final_df_drop`을 생성합니다. 이는 제품 자체에 대한 감성 분석의 정확도를 높이기 위함입니다.
- 필터링 후 `final_df_drop`의 `리뷰분류`별 개수를 확인합니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    # 1. 리뷰분류가 null인 데이터 확인
    null_reviews = final_df[final_df['리뷰분류'].isna()]
    print(f"리뷰분류가 null인 데이터 개수: {len(null_reviews)}개")
    display(null_reviews)

    # 2. 고유한 상품번호 개수 확인
    unique_products = final_df['상품번호'].nunique()
    print(f"\n고유한 상품번호 개수: {unique_products}개")


리뷰분류가 null인 데이터 개수: 0개


,상품번호,피부타입,피부톤,정렬,평점,리뷰내용,리뷰분류



고유한 상품번호 개수: 232개


In [ ]:
if RUN_GEMINI_API:
    pass
    final_df.to_csv(DATA_INTERIM_DIR / "gemini_리뷰감성분석_긍정부정애매.csv", index=False)


In [ ]:
if RUN_GEMINI_API:
    pass
    final_df_pf = final_df[final_df['리뷰분류']!='애매']


In [ ]:
if RUN_GEMINI_API:
    pass
    final_df_pf.reset_index(inplace=True)


In [200]:
if RUN_GEMINI_API:
    pass
    # final_df_pf.drop(columns='index',inplace=True)
    final_df_pf


,상품번호,피부타입,피부톤,정렬,평점,리뷰내용,리뷰분류
0,A000000260257,지성,쿨톤,유용한순,5,너무 만족합니다 좋아요 굿!! 최고입니다 지성피부 잘 맞아요,긍정
1,A000000260257,지성,쿨톤,유용한순,5,자극없이순하고 데이케어용 보습으로 사용하기 좋아요. 화장 전에도 나름 괜찮은 것 같아요,긍정
2,A000000260257,지성,쿨톤,유용한순,5,🌱나 민감성 피부🌱 피부 예민할 때 아무거나 못 바르잖아 이건 그냥 편하게 손이 가는 크림이었음 처음 바를 때 자극 느낌 거의 없고 부드럽게 펴발려서 부담 없이 쓰기 좋았음 막 엄청 꾸덕한 건 아닌데 바르면 촉촉함은 바로 올라오는 편이고 겉돌지 않고 자연스럽게 스며드는 느낌 건조해서 당길 때 발라주면 금방 편안해지는 느낌 들어서 괜찮았음 유분감도 과하지 않아서 지성인데도 크게 답답함 없었고 번들거리기보단 깔끔하게 마무리되는 쪽 와 이거 대박이다 이런 느낌보단 피부 컨디션 안 좋을 때 무난하게 계속 쓰게 되는 그런 크림임,긍정
3,A000000260257,지성,쿨톤,유용한순,5,"피부과에서 추천하는 크림만큼 피부 유수분을 잘맞춰주고 기름짐이 없어요 지성 수부지분들은 정말 만족하실만한 크림 유분이 안느껴지고 끈적거리지 않는 마무리감, 발림성이 너무 좋아요 단점이 없는 크림이라고 생각됩니다. 저는 여름에는 막 써주고 가을 겨울에는 살짝 건조해 레이어드 몇번 해줘 발라줍니다.그럼 전혀 건조해 지지 않아요 제형은 꾸덕하지 않고 수분감이 좀 있는 크림제형이라고 생각하시면 됩니다 흡수도 좋고 화장품 궁합도 잘 맞아 전혀 밀리지 않습니다 이번에 할인할때 많이 쟁이는중",긍정
4,A000000260257,지성,쿨톤,유용한순,5,"예전부터 아토피문제로 피부과 가서 크림 처방 받던게 제로이드 였어요. 타 제품도 받아봤지만 저는 제로이드가 가장 발림성이 깔끔해서 좋더라구요. 올영에 들어와서 완전 신세계인데, 수딩크림이어서 그런지 발림성 굉장히 깔끔하고, 보습과 진정은 잘 돼요! 아침 메이크업 전에도 발라보고 또 한달 후기 남길게요 발림성도 찍어서 올려보아요 도움이 되셨다면 👍🏻",긍정
...,...,...,...,...,...,...,...
49092,A000000260462,트러블성,쿨톤,유용한순,5,수부지라 개기름 올라오는 곳엔 잔뜩 올라오고 건조한 곳은 ㄹㅇ 갈라지고 이래서 고민했는데 매장 직원붐이 추천해주셔서 테스트 하고 바로 구매했어오 .. 기존 아토베리어는 여름에 쓰기엔 너무 부담이엏는데 이건 좋은듯,긍정
49093,A000000260462,트러블성,쿨톤,유용한순,5,에스트라 수딩크림이 안보이더니 리뉴얼되려고 안보였구나~ 똑 떨어져서 사려고 했는데 딱 잘 샀네요. 수딩크림 가벼움 +아토베리어 크림 보습감 섞인것같네요.이제 막 쓰기시작했으니 믿고씁니딘,긍정
49094,A000000260462,트러블성,봄웜톤,유용한순,5,다른 수분크림보다 자극적인거 같아요 비싼데 잘 모르겠어요ㅜ,부정
49095,A000000260462,트러블성,봄웜톤,유용한순,5,순하고 촉촉해서 좋아요!! 유분기 많이 없고 좋음 순해서 추천해요,긍정


In [191]:
if RUN_GEMINI_API:
    pass
    final_df_pf['리뷰분류'].value_counts()


리뷰분류
긍정    46267
부정     2830
Name: count, dtype: int64

In [ ]:
if RUN_GEMINI_API:
    pass
    final_df_pf.to_csv(DATA_INTERIM_DIR / "gemini_리뷰감성분석_긍정부정.csv", index=False)


In [193]:
if RUN_GEMINI_API:
    pass
    shipping_keywords = [
        "배송", "택배", "도착", "발송",
        "포장", "박스", "깨져", "파손",
        "빠르게", "하루만에", "이틀만에"
        ,"일주일만에"
    ]

    mask = ~final_df_pf['리뷰내용'].str.contains(
        "|".join(shipping_keywords),
        na=False
    )

    final_df_drop = final_df_pf[mask]
    # 배송 관련 제외!


In [194]:
if RUN_GEMINI_API:
    pass
    final_df_drop.reset_index(inplace=True)


In [197]:
if RUN_GEMINI_API:
    pass
    # final_df_drop.drop(columns='index',inplace=True)
    final_df_drop


,상품번호,피부타입,피부톤,정렬,평점,리뷰내용,리뷰분류
0,A000000260257,지성,쿨톤,유용한순,5,너무 만족합니다 좋아요 굿!! 최고입니다 지성피부 잘 맞아요,긍정
1,A000000260257,지성,쿨톤,유용한순,5,자극없이순하고 데이케어용 보습으로 사용하기 좋아요. 화장 전에도 나름 괜찮은 것 같아요,긍정
2,A000000260257,지성,쿨톤,유용한순,5,🌱나 민감성 피부🌱 피부 예민할 때 아무거나 못 바르잖아 이건 그냥 편하게 손이 가는 크림이었음 처음 바를 때 자극 느낌 거의 없고 부드럽게 펴발려서 부담 없이 쓰기 좋았음 막 엄청 꾸덕한 건 아닌데 바르면 촉촉함은 바로 올라오는 편이고 겉돌지 않고 자연스럽게 스며드는 느낌 건조해서 당길 때 발라주면 금방 편안해지는 느낌 들어서 괜찮았음 유분감도 과하지 않아서 지성인데도 크게 답답함 없었고 번들거리기보단 깔끔하게 마무리되는 쪽 와 이거 대박이다 이런 느낌보단 피부 컨디션 안 좋을 때 무난하게 계속 쓰게 되는 그런 크림임,긍정
3,A000000260257,지성,쿨톤,유용한순,5,"피부과에서 추천하는 크림만큼 피부 유수분을 잘맞춰주고 기름짐이 없어요 지성 수부지분들은 정말 만족하실만한 크림 유분이 안느껴지고 끈적거리지 않는 마무리감, 발림성이 너무 좋아요 단점이 없는 크림이라고 생각됩니다. 저는 여름에는 막 써주고 가을 겨울에는 살짝 건조해 레이어드 몇번 해줘 발라줍니다.그럼 전혀 건조해 지지 않아요 제형은 꾸덕하지 않고 수분감이 좀 있는 크림제형이라고 생각하시면 됩니다 흡수도 좋고 화장품 궁합도 잘 맞아 전혀 밀리지 않습니다 이번에 할인할때 많이 쟁이는중",긍정
4,A000000260257,지성,쿨톤,유용한순,5,"예전부터 아토피문제로 피부과 가서 크림 처방 받던게 제로이드 였어요. 타 제품도 받아봤지만 저는 제로이드가 가장 발림성이 깔끔해서 좋더라구요. 올영에 들어와서 완전 신세계인데, 수딩크림이어서 그런지 발림성 굉장히 깔끔하고, 보습과 진정은 잘 돼요! 아침 메이크업 전에도 발라보고 또 한달 후기 남길게요 발림성도 찍어서 올려보아요 도움이 되셨다면 👍🏻",긍정
...,...,...,...,...,...,...,...
47482,A000000260462,트러블성,쿨톤,유용한순,5,수부지라 개기름 올라오는 곳엔 잔뜩 올라오고 건조한 곳은 ㄹㅇ 갈라지고 이래서 고민했는데 매장 직원붐이 추천해주셔서 테스트 하고 바로 구매했어오 .. 기존 아토베리어는 여름에 쓰기엔 너무 부담이엏는데 이건 좋은듯,긍정
47483,A000000260462,트러블성,쿨톤,유용한순,5,에스트라 수딩크림이 안보이더니 리뉴얼되려고 안보였구나~ 똑 떨어져서 사려고 했는데 딱 잘 샀네요. 수딩크림 가벼움 +아토베리어 크림 보습감 섞인것같네요.이제 막 쓰기시작했으니 믿고씁니딘,긍정
47484,A000000260462,트러블성,봄웜톤,유용한순,5,다른 수분크림보다 자극적인거 같아요 비싼데 잘 모르겠어요ㅜ,부정
47485,A000000260462,트러블성,봄웜톤,유용한순,5,순하고 촉촉해서 좋아요!! 유분기 많이 없고 좋음 순해서 추천해요,긍정


In [198]:
if RUN_GEMINI_API:
    pass
    final_df_drop['리뷰분류'].value_counts()


리뷰분류
긍정    44707
부정     2780
Name: count, dtype: int64

## 13. 최종 데이터 저장

- `final_df_drop` 데이터프레임을 `gemini_리뷰감성분석_긍정부정_배송제외.csv` 파일로 저장합니다. 이 파일은 '애매' 리뷰와 배송 관련 리뷰가 제외된 최종 감성 분석 결과를 포함합니다.

In [ ]:
if RUN_GEMINI_API:
    pass
    final_df_drop.to_csv(DATA_PROCESSED_DIR / "gemini_리뷰감성분석_긍정부정_배송제외.csv", index=False)


In [201]:
if RUN_GEMINI_API:
    pass
    #고유한 상품번호 개수 확인
    unique_products_final = final_df_drop['상품번호'].nunique()
    print(f"\n고유한 상품번호 개수: {unique_products}개")



고유한 상품번호 개수: 232개


## 저장된 Gemini 감성분석 결과

In [ ]:
import pandas as pd

SAVED_RESULT_PATH = (
    DATA_PROCESSED_DIR
    / "Gemini_리뷰감성분석_배송제외_최종.csv"
)

sentiment_result = pd.read_csv(SAVED_RESULT_PATH, encoding="utf-8-sig")
print("저장된 감성분석 결과:", sentiment_result.shape)
display(sentiment_result["리뷰분류_텍스트"].value_counts(dropna=False))
display(sentiment_result.head())
